# Day 2 — Data Cleaning + SQL Database Design
**Bluestock Capstone I: Mutual Fund Analytics**


---
## 0. Setup

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sqlalchemy import create_engine, text
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = Path().resolve().parent
RAW      = BASE_DIR / 'data' / 'raw'
PROC     = BASE_DIR / 'data' / 'processed'
DB_PATH  = BASE_DIR / 'data' / 'db' / 'bluestock_mf.db'

print('Project root:', BASE_DIR)
files=[f.name for f in RAW.glob('*.csv') if f.name[0].isdigit()]
print(files)

Project root: /home/l011yp0p/data/intership/bluestock_mf_capstone
['03_aum_by_fund_house.csv', '05_category_inflows.csv', '06_industry_folio_count.csv', '01_fund_master.csv', '07_scheme_performance.csv', '04_monthly_sip_inflows.csv', '08_investor_transactions.csv', '10_benchmark_indices.csv', '02_nav_history.csv', '09_portfolio_holdings.csv']


In [32]:
# Displaying all csv files and columns names
for f in files:
    df=pd.read_csv(RAW/f)
    print(f, df.columns.tolist())

03_aum_by_fund_house.csv ['date', 'fund_house', 'aum_lakh_crore', 'aum_crore', 'num_schemes']
05_category_inflows.csv ['month', 'category', 'net_inflow_crore']
06_industry_folio_count.csv ['month', 'total_folios_crore', 'equity_folios_crore', 'debt_folios_crore', 'hybrid_folios_crore', 'others_folios_crore']
01_fund_master.csv ['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category', 'plan', 'launch_date', 'benchmark', 'expense_ratio_pct', 'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager', 'risk_category', 'sebi_category_code']
07_scheme_performance.csv ['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade']
04_monthly_sip_inflows.csv ['month', 'sip_inflow_crore', 'active_sip_accounts_crore', 'new_sip_accounts_lakh

---
## Clean nav_history.csv

In [7]:
# Load raw file
nav_raw = pd.read_csv(RAW / '02_nav_history.csv')
print('Raw shape:', nav_raw.shape)
print(nav_raw.dtypes)
nav_raw.head(5)

Raw shape: (46000, 3)
amfi_code      int64
date          object
nav          float64
dtype: object


,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [9]:
# Parse dates
nav = nav_raw.copy()
nav['date'] = pd.to_datetime(nav['date'], errors='coerce')
print('Unparseable dates:', nav['date'].isna().sum())
nav = nav.dropna(subset=['date'])

# Remove duplicates
before = len(nav)
nav = nav.drop_duplicates(subset=['amfi_code','date'])
print(f'Duplicates removed: {before - len(nav)}')

# Sort
nav = nav.sort_values(['amfi_code','date']).reset_index(drop=True)

# Forward-fill missing NAV for holidays
all_bdays = pd.bdate_range(nav['date'].min(), nav['date'].max())
filled = []
for code, grp in nav.groupby('amfi_code'):
    grp = grp.set_index('date').reindex(all_bdays)
    grp['amfi_code'] = code
    grp['nav'] = grp['nav'].ffill()
    grp.index.name = 'date'
    filled.append(grp.reset_index())
nav = pd.concat(filled, ignore_index=True).dropna(subset=['nav'])

# Validate NAV > 0
print('NAV <= 0:', (nav['nav'] <= 0).sum())
nav = nav[nav['nav'] > 0]

# Daily return
nav['daily_return_pct'] = nav.groupby('amfi_code')['nav'].pct_change() * 100

print(f'Clean shape: {nav.shape}')
nav.to_csv(PROC / 'clean_nav_history.csv', index=False)
nav.head(3)

Unparseable dates: 0
Duplicates removed: 0
NAV <= 0: 0
Clean shape: (46000, 4)


,date,amfi_code,nav,daily_return_pct
0,2022-01-03,100016,520.4608,NaN
1,2022-01-04,100016,515.0971,-1.030568
2,2022-01-05,100016,521.7239,1.286515


---
## Clean investor_transactions.csv

In [11]:
tx_raw = pd.read_csv(RAW / '08_investor_transactions.csv')
print('Raw shape:', tx_raw.shape)
print('\ntransaction_type values:')
print(tx_raw['transaction_type'].value_counts())
print('\nkyc_status values:')
print(tx_raw['kyc_status'].value_counts())

Raw shape: (32778, 13)

transaction_type values:
transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64

kyc_status values:
kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64


In [14]:
tx = tx_raw.copy()

# Standardise transaction_type
TX_MAP = {'sip':'SIP','Sip':'SIP','SIP':'SIP',
          'lumpsum':'Lumpsum','LUMPSUM':'Lumpsum','Lumpsum':'Lumpsum',
          'redemption':'Redemption','REDEMPTION':'Redemption','Redemption':'Redemption',
          'stp':'STP','STP':'STP'}
tx['transaction_type'] = tx['transaction_type'].map(TX_MAP).fillna('Other')
print('After standardising:', tx['transaction_type'].value_counts().to_dict())

# Parse date (DD/MM/YYYY)
tx['transaction_date'] = pd.to_datetime(tx['transaction_date'], dayfirst=True, errors='coerce')
print('Bad dates:', tx['transaction_date'].isna().sum())
tx = tx.dropna(subset=['transaction_date'])

# Validate amount > 0
neg = (tx['amount_inr'] <= 0).sum()
print(f'Negative amounts removed: {neg}')
tx = tx[tx['amount_inr'] > 0]

# KYC status
KYC_MAP = {'KYC Verified':'KYC Verified','kyc_verified':'KYC Verified',
           'VERIFIED':'KYC Verified','Pending':'Pending','pending':'Pending','N/A':'Pending'}
tx['kyc_status'] = tx['kyc_status'].map(KYC_MAP).fillna('Pending')

tx.to_csv(PROC / 'clean_transactions.csv', index=False)
print(f'Clean shape: {tx.shape}')
tx.head(3)

After standardising: {'SIP': 19716, 'Lumpsum': 8095, 'Redemption': 4967}
Bad dates: 19730
Negative amounts removed: 0
Clean shape: (13048, 13)


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Pending
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Pending
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Pending


---
## Clean scheme_performance.csv

In [18]:
perf_raw = pd.read_csv(RAW / '07_scheme_performance.csv')
print('Raw shape:', perf_raw.shape)
print(perf_raw.dtypes)
perf_raw.head(5)

Raw shape: (40, 19)
amfi_code               int64
scheme_name            object
fund_house             object
category               object
plan                   object
return_1yr_pct        float64
return_3yr_pct        float64
return_5yr_pct        float64
benchmark_3yr_pct     float64
alpha                 float64
beta                  float64
sharpe_ratio          float64
sortino_ratio         float64
std_dev_ann_pct       float64
max_drawdown_pct      float64
aum_crore               int64
expense_ratio_pct     float64
morningstar_rating      int64
risk_grade             object
dtype: object


,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


In [25]:
perf = perf_raw.copy()

# Coerce all return columns to numeric
ret_cols = ['return_1yr_pct','return_3yr_pct','return_5yr_pct','sharpe_ratio','beta',
            'alpha','std_dev_ann_pct','sortino_ratio','max_drawdown_pct']
for col in ret_cols:
    if col in perf.columns:
        perf[col] = pd.to_numeric(perf[col], errors='coerce')
        nulls = perf[col].isna().sum()
        if nulls: print(f'  {col}: {nulls} non-numeric → NaN')

# Flag negative Sharpe
perf['sharpe_flag'] = perf['sharpe_ratio'].apply(
    lambda x: 'NEGATIVE' if pd.notna(x) and x < 0 else 'OK'
)
print('Negative Sharpe:', (perf['sharpe_flag']=='NEGATIVE').sum())

# Check expense_ratio range
out = ~perf['expense_ratio_pct'].between(0.1, 2.5)
print(f'expense_ratio out of range: {out.sum()} → set to NaN')
perf.loc[out, 'expense_ratio_pct'] = np.nan

# Clip max drawdown
perf['max_drawdown_pct'] = perf['max_drawdown_pct'].clip(-1.0, 0.0)

perf.to_csv(PROC / 'clean_performance.csv', index=False)
print(f'Clean shape: {perf.shape}')
perf[['return_1yr_pct','return_3yr_pct','sharpe_ratio','expense_ratio_pct']].describe().round(3)

Negative Sharpe: 0
expense_ratio out of range: 0 → set to NaN
Clean shape: (40, 20)


,return_1yr_pct,return_3yr_pct,sharpe_ratio,expense_ratio_pct
count,40.000,40.000,40.000,40.000
mean,14.376,14.089,1.362,1.237
std,4.883,4.617,1.476,0.387
min,4.260,5.140,0.800,0.550
25%,11.735,12.035,0.865,0.788
50%,14.620,14.205,0.925,1.425
75%,16.393,15.882,0.985,1.540
max,24.930,23.390,7.680,1.640


---
## Cleaning other csv files using scripts/data_cleaning.py

In [47]:
print('Project root:', BASE_DIR)
processed=[f.name for f in PROC.glob('*.csv')]
for f in processed:
    df=pd.read_csv(PROC/f)
    print(f, df.columns.tolist())

Project root: /home/l011yp0p/data/intership/bluestock_mf_capstone
clean_benchmark_indicies.csv ['date', 'index_name', 'close_value']
clean_category_inflows.csv ['month', 'category', 'net_inflow_crore']
clean_industry_folio_count.csv ['month', 'total_folios_crore', 'equity_folios_crore', 'debt_folios_crore', 'hybrid_folios_crore', 'others_folios_crore']
clean_monthly_sip_inflows.csv ['month', 'sip_inflow_crore', 'active_sip_accounts_crore', 'new_sip_accounts_lakh', 'sip_aum_lakh_crore', 'yoy_growth_pct']
clean_fund_master.csv ['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category', 'plan', 'launch_date', 'benchmark', 'expense_ratio_pct', 'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager', 'risk_category', 'sebi_category_code']
clean_aum_by_fund_house.csv ['date', 'fund_house', 'aum_lakh_crore', 'aum_crore', 'num_schemes']
clean_nav_history.csv ['date', 'amfi_code', 'nav', 'daily_return_pct']
clean_scheme_performance.csv ['amfi_code', 'scheme_name', 'fu

---
## SQLite Schema Design

In [91]:
# Print the schema
schema = (BASE_DIR / 'sql' / 'schema.sql').read_text()
print(schema[:3500])

-- ============================================================
-- schema.sql
-- Bluestock MF Capstone I — SQLite Star Schema
-- Day 2 — Task 4
--
-- Tables (matching official spec):
--   dim_fund           40 rows
--   dim_date         1500 rows
--   fact_nav        46000 rows
--   fact_transactions 32000+ rows
--   fact_performance    40 rows
--   fact_portfolio     320 rows
--   fact_aum            90 rows
--   fact_sip_industry   48 rows
--
-- Run:
--   sqlite3 data/db/bluestock_mf.db < sql/schema.sql
-- ============================================================

PRAGMA foreign_keys = ON;
PRAGMA journal_mode = WAL;


-- ============================================================
-- DIMENSION TABLES
-- ============================================================

CREATE TABLE IF NOT EXISTS dim_fund (
    amfi_code       TEXT    PRIMARY KEY,
    scheme_name     TEXT    NOT NULL,
    fund_house      TEXT    NOT NULL,
    category        TEXT,
    sub_category    TEXT,
    risk_grad

---
## Load into SQLite

In [92]:
import subprocess
result = subprocess.run(
    ['python', str(BASE_DIR / 'scripts' / 'db_loader.py'), '--fresh'],
    capture_output=True, text=True, cwd=str(BASE_DIR)
)
print(result.stdout)
if result.returncode != 0:
    print('ERROR:', result.stderr)

  Bluestock MF — DB Loader (Day 2)
  DB: /home/l011yp0p/data/intership/bluestock_mf_capstone/data/db/bluestock_mf.db
  All tables dropped.

  Applying schema from schema.sql ...
  Schema applied.

  Loading dimension tables ...
  clean_fund_master.csv               → dim_fund                         40 rows

  Building dim_date ...
  dim_date loaded: 1,608 rows

  Loading fact tables ...
  clean_nav_history.csv               → fact_nav                     46,000 rows
  clean_investor_transactions.csv     → fact_transactions            13,048 rows
  clean_scheme_performance.csv        → fact_performance                 40 rows
  clean_portfolio_holdings.csv        → fact_portfolio                  322 rows
  clean_aum_by_fund_house.csv         → fact_aum                         90 rows
  clean_monthly_sip_inflows.csv       → fact_sip_industry                36 rows
  clean_benchmark_indicies.csv        → fact_benchmark_indices        8,050 rows
  clean_category_inflows.csv          → fa

In [93]:
# Verify via SQLAlchemy
engine = create_engine(f'sqlite:///{DB_PATH}')
tables = ['dim_fund','dim_date','fact_nav','fact_transactions',
          'fact_performance','fact_portfolio','fact_aum','fact_sip_industry']
print('Table'.ljust(28), 'Rows')
print('-' * 38)
for t in tables:
    with engine.connect() as conn:
        n = conn.execute(text(f'SELECT COUNT(*) FROM {t}')).scalar()
    print(t.ljust(28), f'{n:>8,}')

Table                        Rows
--------------------------------------
dim_fund                           40
dim_date                        1,608
fact_nav                       46,000
fact_transactions              13,048
fact_performance                   40
fact_portfolio                    322
fact_aum                           90
fact_sip_industry                  36


---
## SQL Analytics Queries

In [94]:
def run_query(sql, title=''):
    with engine.connect() as conn:
        df = pd.read_sql(text(sql), conn)
    print(f'\n── {title}')
    return df

In [95]:
q1 = run_query("""
SELECT scheme_name, fund_house, ROUND(aum_crore,2) AS aum_crore
FROM dim_fund ORDER BY aum_crore DESC LIMIT 5
""", 'Top 5 funds by AUM')
q1


── Top 5 funds by AUM


,scheme_name,fund_house,aum_crore
0,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,1250000.0
1,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,1250000.0
2,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,1250000.0
3,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,1250000.0
4,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,1250000.0


In [100]:
q2 = run_query("""
SELECT strftime('%Y-%m', date) AS month, ROUND(AVG(nav),2) AS avg_nav
FROM fact_nav GROUP BY month ORDER BY month
""", 'Avg NAV per month')
q2.head(5)


── Avg NAV per month


,month,avg_nav
0,2022-01,207.06
1,2022-02,207.72
2,2022-03,209.69
3,2022-04,211.83
4,2022-05,212.73


In [101]:
q3 = run_query("""
SELECT strftime('%Y', month) AS year,
       ROUND(SUM(sip_inflow_crore),2) AS total_inflow_crore
FROM fact_sip_industry GROUP BY year ORDER BY year
""", 'SIP YoY growth')
q3


── SIP YoY growth


,year,total_inflow_crore
0,2023,184763.0
1,2024,269781.0
2,2025,335740.0


In [102]:
q4 = run_query("""
SELECT state, COUNT(*) AS txns, ROUND(SUM(amount),2) AS total_rs
FROM fact_transactions WHERE state IS NOT NULL
GROUP BY state ORDER BY total_rs DESC
""", 'Transactions by state')
q4


── Transactions by state


,state,txns,total_rs
0,Tamil Nadu,1100,126020641.0
1,Madhya Pradesh,1137,124230586.0
2,West Bengal,1130,121974095.0
3,Punjab,1164,121295845.0
4,Uttar Pradesh,1076,119497728.0
5,Gujarat,1113,117945641.0
6,Delhi,1083,116939568.0
7,Rajasthan,1010,114352455.0
8,Karnataka,1057,113816812.0
9,Haryana,1096,112280777.0


In [103]:
q5 = run_query("""
SELECT d.scheme_name, d.fund_house, ROUND(p.expense_ratio,2) AS er
FROM dim_fund d JOIN fact_performance p ON d.amfi_code=p.amfi_code
WHERE p.expense_ratio < 1.0 AND p.expense_ratio IS NOT NULL
ORDER BY er
""", 'Funds with expense_ratio < 1%')
q5


── Funds with expense_ratio < 1%


,scheme_name,fund_house,er
0,Nippon India Gilt Securities Fund - Regular - ...,Nippon India MF,0.55
1,HDFC Short Term Debt Fund - Regular - Growth,HDFC Mutual Fund,0.56
2,Kotak Liquid Fund - Regular - Growth,Kotak Mahindra MF,0.60
3,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,0.66
4,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,0.72
5,Nippon India Large Cap Fund - Direct - Growth,Nippon India MF,0.72
6,ICICI Pru Liquid Fund - Regular - Growth,ICICI Prudential MF,0.74
7,Axis Bluechip Fund - Direct - Growth,Axis Mutual Fund,0.75
8,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,0.77
9,HDFC Mid-Cap Opportunities Fund - Direct - Growth,HDFC Mutual Fund,0.78


In [104]:
# Q6-Q10 — run via scripts
import subprocess
result = subprocess.run(['python', str(BASE_DIR/'scripts'/'run_queries.py')],
                        capture_output=True, text=True, cwd=str(BASE_DIR))
print(result.stdout[-3000:])
print('Query CSVs saved to reports/query_results/')

utual Fund 2024-09  1080000.0          186
         SBI Mutual Fund 2024-12  1114000.0          186
         SBI Mutual Fund 2025-03  1250000.0          186
         SBI Mutual Fund 2025-12  1250000.0          186
         UTI Mutual Fund 2022-03   230000.0          142
         UTI Mutual Fund 2022-09   232000.0          142
         UTI Mutual Fund 2023-03   239000.0          142
         UTI Mutual Fund 2023-09   265000.0          142
         UTI Mutual Fund 2024-03   290000.0          142
         UTI Mutual Fund 2024-09   308000.0          142
         UTI Mutual Fund 2024-12   352000.0          142
         UTI Mutual Fund 2025-03   355000.0          142
         UTI Mutual Fund 2025-12   410000.0          142

  → 90 rows returned
  → Saved: reports/query_results/q8_aum_trend_per_fund_house_over_time.csv

────────────────────────────────────────────────────────────
  Q9. Top 10 most held stocks across all portfolios
────────────────────────────────────────────────────────────
s